# 03_indice_oportunidad

## Objetivo
Calcular el índice de oportunidad de microcrédito digital por departamento a partir del dataset maestro.

## Alcance
- Cargar `master_dataset.csv`
- Normalizar variables
- Definir pesos
- Calcular índice
- Generar ranking
- Explorar escenarios (opcional)

## 1. Librerías

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

## 2. Rutas y carga de datos

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(DATA_PROCESSED / 'master_dataset.csv')
df.head()

,departamento,pobreza_2024,acceso_microcredito_2024,acceso_productos_financieros_2024,atm_x_10000_adultos_2024,internet_hogares_2024
0,AMAZONAS,NaN,3.4,67.0,2.5,41.3
1,ANTIOQUIA,24.7,4.4,126.3,4.6,66.9
2,ARAUCA,NaN,7.9,78.1,2.1,34.3
3,ATLÁNTICO,31.6,2.8,94.2,4.5,58.9
4,BOGOTÁ D.C.,19.6,2.9,118.8,7.5,82.7


## 3. Revisión inicial

In [3]:
print(df.shape)
print(df.isna().sum())
df.describe()

(37, 6)
departamento                          0
pobreza_2024                         13
acceso_microcredito_2024              4
acceso_productos_financieros_2024     4
atm_x_10000_adultos_2024              4
internet_hogares_2024                 4
dtype: int64


,pobreza_2024,acceso_microcredito_2024,acceso_productos_financieros_2024,atm_x_10000_adultos_2024,internet_hogares_2024
count,24.000000,33.000000,33.000000,33.000000,33.000000
mean,37.166667,6.593939,80.309091,3.345455,55.587879
std,14.188840,3.407339,20.999514,1.697073,16.788161
min,19.600000,2.800000,20.400000,1.200000,15.500000
25%,25.450000,3.900000,71.700000,2.100000,46.400000
50%,35.450000,5.600000,81.600000,3.300000,60.200000
75%,47.850000,9.000000,93.000000,4.000000,66.900000
max,67.400000,13.400000,126.300000,8.800000,82.700000


## 4. Normalización de variables
Se utiliza Min-Max scaling para llevar todas las variables al rango [0,1].

In [4]:
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

df_norm = df.copy()

df_norm['pobreza_n'] = minmax(df['pobreza_2024'])
df_norm['microcredito_n'] = 1 - minmax(df['acceso_microcredito_2024'])
df_norm['productos_n'] = 1 - minmax(df['acceso_productos_financieros_2024'])
df_norm['atm_n'] = 1 - minmax(df['atm_x_10000_adultos_2024'])
df_norm['internet_n'] = minmax(df['internet_hogares_2024'])

df_norm[['departamento','pobreza_n','microcredito_n','productos_n','atm_n','internet_n']].head()

,departamento,pobreza_n,microcredito_n,productos_n,atm_n,internet_n
0,AMAZONAS,NaN,0.943396,0.559962,0.828947,0.383929
1,ANTIOQUIA,0.106695,0.849057,0.000000,0.552632,0.764881
2,ARAUCA,NaN,0.518868,0.455146,0.881579,0.279762
3,ATLÁNTICO,0.251046,1.000000,0.303116,0.565789,0.645833
4,BOGOTÁ D.C.,0.000000,0.990566,0.070822,0.171053,1.000000


## 5. Definición de pesos
Los pesos representan la importancia relativa de cada dimensión.

In [5]:
pesos = {
    'pobreza_n': 0.30,
    'microcredito_n': 0.30,
    'productos_n': 0.15,
    'atm_n': 0.10,
    'internet_n': 0.15
}

pesos

{'pobreza_n': 0.3,
 'microcredito_n': 0.3,
 'productos_n': 0.15,
 'atm_n': 0.1,
 'internet_n': 0.15}

## 6. Cálculo del índice de oportunidad

In [6]:
df_norm['indice_oportunidad'] = (
    df_norm['pobreza_n'] * pesos['pobreza_n'] +
    df_norm['microcredito_n'] * pesos['microcredito_n'] +
    df_norm['productos_n'] * pesos['productos_n'] +
    df_norm['atm_n'] * pesos['atm_n'] +
    df_norm['internet_n'] * pesos['internet_n']
)

df_norm[['departamento','indice_oportunidad']].head()

,departamento,indice_oportunidad
0,AMAZONAS,NaN
1,ANTIOQUIA,0.456721
2,ARAUCA,NaN
3,ATLÁNTICO,0.574235
4,BOGOTÁ D.C.,0.474898


## 7. Ranking de departamentos

In [7]:
ranking = df_norm.sort_values(by='indice_oportunidad', ascending=False).reset_index(drop=True)
ranking[['departamento','indice_oportunidad']].head(10)

,departamento,indice_oportunidad
0,CHOCÓ,0.798533
1,LA GUAJIRA,0.795818
2,MAGDALENA,0.677146
3,SUCRE,0.666435
4,BOLÍVAR,0.661740
5,CÓRDOBA,0.642076
6,ATLÁNTICO,0.574235
7,NORTE DE SANTANDER,0.538886
8,VALLE DEL CAUCA,0.510571
9,CAUCA,0.506240


## 8. Clasificación por niveles

In [8]:
df_norm['nivel'] = pd.qcut(df_norm['indice_oportunidad'], q=3, labels=['Bajo','Medio','Alto'])
df_norm[['departamento','indice_oportunidad','nivel']].sort_values(by='indice_oportunidad', ascending=False).head(10)

,departamento,indice_oportunidad,nivel
12,CHOCÓ,0.798533,Alto
19,LA GUAJIRA,0.795818,Alto
20,MAGDALENA,0.677146,Alto
32,SUCRE,0.666435,Alto
5,BOLÍVAR,0.661740,Alto
15,CÓRDOBA,0.642076,Alto
3,ATLÁNTICO,0.574235,Alto
23,NORTE DE SANTANDER,0.538886,Alto
34,VALLE DEL CAUCA,0.510571,Medio
10,CAUCA,0.506240,Medio


## 9. Guardar resultados

In [9]:
output = DATA_PROCESSED / 'ranking_oportunidad.csv'
df_norm.to_csv(output, index=False)
print('Guardado en:', output)

Guardado en: C:\Users\guill\microAI\data\processed\ranking_oportunidad.csv


## 10. Exploración de escenarios (opcional)
Permite modificar pesos para observar cambios en el ranking.

In [10]:
# ejemplo de escenario alternativo
pesos_alt = pesos.copy()
pesos_alt['internet_n'] = 0.30
pesos_alt['pobreza_n'] = 0.20

df_norm['indice_alt'] = (
    df_norm['pobreza_n'] * pesos_alt['pobreza_n'] +
    df_norm['microcredito_n'] * pesos_alt['microcredito_n'] +
    df_norm['productos_n'] * pesos_alt['productos_n'] +
    df_norm['atm_n'] * pesos_alt['atm_n'] +
    df_norm['internet_n'] * pesos_alt['internet_n']
)

df_norm.sort_values(by='indice_alt', ascending=False)[['departamento','indice_alt']].head(10)

,departamento,indice_alt
19,LA GUAJIRA,0.731964
12,CHOCÓ,0.727998
5,BOLÍVAR,0.702102
20,MAGDALENA,0.698384
32,SUCRE,0.656120
15,CÓRDOBA,0.650074
3,ATLÁNTICO,0.646006
34,VALLE DEL CAUCA,0.627050
4,BOGOTÁ D.C.,0.624898
26,RISARALDA,0.619674
